# Proteome exploration with the **full SAE feature vector**

Companion to `explore_proteome_sae.ipynb`. Identical pipeline — one scanpy graph driving a UMAP layout and Leiden clusters, per-cluster SAE-feature enrichment, ESM Atlas annotation — **except the KNN graph is built on the FULL pooled SAE activation vector** for each protein (every non-zero feature, ~1,800 median per protein), read from the local `SaeFeatureStore` `.npz`, rather than the top-K=64 summary in `sae_top_features`.

**Prerequisite:** the full-vector store (`data/sae_feature_matrix.npz`) must be populated for the proteins of interest (embed/SAE run with `sae.store_full`). Proteins missing from the store are dropped.

In [ ]:
from och_annotate.config import load_config
from och_annotate.analysis import load_embeddings

cfg = load_config("../config/octopus_chierchiae.yaml")
df = load_embeddings(cfg, prefer_cache=True)   # backfills Baserow metadata cols not in cache
print(f"Loaded {len(df)} embeddings; vector dim = {len(df['embedding'].iloc[0])}")
df.head()

## Clustering basis: the full SAE vector store

This notebook reads the complete pooled SAE vectors from `data/sae_feature_matrix.npz` and uses them directly as the KNN/UMAP/Leiden basis. Where the top-K notebook clusters on 64 features per protein, this one uses all non-zero pooled features (~1,800 median), so cluster structure reflects the full activation profile rather than just each protein's strongest features.

In [ ]:
import numpy as np
from och_annotate.analysis import SaeFeatureStore

store = SaeFeatureStore("../data/sae_feature_matrix.npz")
m = store.to_csr(); nnz = np.asarray((m > 0).sum(axis=1)).ravel()
print(f"Full SAE store: {len(store)} proteins x {store.n_features} features "
      f"(model {store.sae_model})")
print(f"  active features/protein: min {int(nnz.min())}, median "
      f"{int(np.median(nnz))}, max {int(nnz.max())} -> clustering on these FULL vectors")

## One graph: KNN → UMAP → Leiden (on the FULL SAE vector)

`build_anndata_full` loads the complete pooled SAE vectors from the store as `adata.X` (proteins × 16,384), aligned to the metadata by `transcript_id`. The neighbor graph is built on an IDF-weighted version of that full matrix, then one UMAP layout and Leiden clusters come off it.

Parameters are inherited from the top-K notebook (TF-IDF / cosine / `n_neighbors=15` / `resolution=2.0`). Those were tuned for the *top-K* space; the full-vector space is far denser, so they likely warrant re-tuning — expect a different cluster count.

In [ ]:
import numpy as np
import scanpy as sc
from scipy import sparse
from och_annotate.analysis import build_anndata_full, sae_enrichment, plot_umap
import plotly.io as pio
pio.renderers.default = "notebook"   # embed interactive plots into nbconvert HTML

# Params inherited from the top-K notebook; tuned for the top-K space, so the
# full-vector space (much denser) likely warrants its own sweep.
SEED        = 0
N_NEIGHBORS = 15
MIN_DIST    = 0.3       # UMAP layout spread only (not used for the graph/clustering)
METRIC      = "cosine"
LEIDEN_RES  = 2.0       # granularity dial (cluster count differs for the full-vector space)

adata = build_anndata_full(df, "../data/sae_feature_matrix.npz")  # X = FULL pooled SAE vectors

# Cluster on IDF-weighted FULL SAE activations: rare, protein-family-specific
# features upweighted, ubiquitous ones damped. Raw activations (adata.X) are
# left untouched so downstream enrichment stays honest.
df_count = np.asarray((adata.X > 0).sum(axis=0)).ravel()
idf = np.log((adata.n_obs + 1) / (df_count + 1)) + 1.0
adata.obsm["X_sae_tfidf"] = adata.X.multiply(sparse.csr_matrix(idf)).tocsr()

sc.pp.neighbors(adata, use_rep="X_sae_tfidf", n_neighbors=N_NEIGHBORS, metric=METRIC, random_state=SEED)
sc.tl.umap(adata, min_dist=MIN_DIST, random_state=SEED)
sc.tl.leiden(adata, resolution=LEIDEN_RES, flavor="igraph", n_iterations=2,
             directed=False, random_state=SEED)

coords = adata.obs.reset_index(drop=True).copy()
coords["umap_0"] = adata.obsm["X_umap"][:, 0]
coords["umap_1"] = adata.obsm["X_umap"][:, 1]
coords["leiden"] = adata.obs["leiden"].to_numpy()
print(f"{adata.n_obs} proteins; {coords['leiden'].nunique()} Leiden clusters")

In [ ]:
from och_annotate.analysis import plot_umap_searchable

# UMAP colored by chromosome, with a client-side gene search box (works in the
# exported HTML). Searches gene/ortholog/orthogroup ids; matches are ringed.
# embed_js=True here loads plotly.js once for the whole document.
plot_umap_searchable(coords, color="chromosome",
          hover=["transcript_id", "gene_id", "Ochierchiae_name", "Mmusculus_gene_name", "orthogroup", "chromosome", "leiden"],
          search_fields=["transcript_id", "gene_id", "Ochierchiae_name", "Mmusculus_gene_name", "orthogroup"],
          labels={"orthogroup": "Orthogroup"},
          title=f"{cfg.name} — UMAP on full SAE features (chromosome)", embed_js=True)

In [ ]:
# Same UMAP colored by Leiden cluster, same searchable box. embed_js=False:
# reuse the plotly.js already loaded above (avoids embedding it twice).
plot_umap_searchable(coords, color="leiden",
          hover=["transcript_id", "gene_id", "Ochierchiae_name", "Mmusculus_gene_name", "orthogroup", "chromosome", "leiden"],
          search_fields=["transcript_id", "gene_id", "Ochierchiae_name", "Mmusculus_gene_name", "orthogroup"],
          labels={"orthogroup": "Orthogroup"},
          title=f"{cfg.name} — UMAP on full SAE features (Leiden clusters)", embed_js=False)

## SAE-feature enrichment per cluster

Wilcoxon rank-sum on the SAE activation matrix, annotated from the **ESM Atlas**: `label`, `category`, `activation_pattern`, `exemplar_protein_families`, top **SwissProt** proteins, and `uniref90_idf` — used to **IDF-weight** markers toward specific (rare) features.

In [ ]:
import pandas as pd
from och_annotate.atlas import fetch_feature_descriptions

enrich = sae_enrichment(adata, groupby="leiden", method="wilcoxon", n=15)

# Rich per-feature Atlas metadata for the enriched features (concurrent, cached; no credits)
feat_ids = sorted(enrich["sae_feature"].astype(int).unique())
meta = fetch_feature_descriptions(feat_ids, cache_path="../data/sae_feature_metadata.parquet")
keep = ["feature", "label", "category", "activation_pattern",
        "exemplar_protein_families", "uniref90_idf", "swissprot_top"]
meta = meta[keep].copy(); meta["feature"] = meta["feature"].astype(str)

enrich["feature"] = enrich["sae_feature"].astype(str)
enrich = enrich.merge(meta, on="feature", how="left").drop(columns="feature")

# IDF-weighting: upweight features that are rare across UniRef90 (more specific).
enrich["idf"] = pd.to_numeric(enrich["uniref90_idf"], errors="coerce").fillna(1.0)
enrich["score_idf"] = enrich["scores"] * enrich["idf"]

enrich.to_csv("../data/cluster_sae_enrichment_fullsae.csv", index=False)
print(f"Annotated {len(feat_ids)} features (category / IDF / exemplars / SwissProt); "
      f"{len(enrich)} rows across {enrich['leiden'].nunique()} clusters")

In [ ]:
# Per-cluster functional profile: the category mix of each cluster's top-15 features
profile = (enrich.assign(cluster=enrich["leiden"].astype(int))
                 .groupby("cluster")["category"]
                 .apply(lambda s: ", ".join(f"{c} ({n})" for c, n in
                        s.fillna("(uncat)").replace("", "(uncat)").value_counts().head(4).items()))
                 .rename("top_feature_categories").to_frame())
with pd.option_context("display.max_rows", None, "display.max_colwidth", 90):
    display(profile)

In [ ]:
from IPython.display import display

# Top-5 per cluster, ranked by the IDF-WEIGHTED score (specific features rise).
top5 = (enrich.assign(cluster=enrich["leiden"].astype(int))
              .sort_values(["cluster", "score_idf"], ascending=[True, False])
              .groupby("cluster", observed=True).head(5).copy())
top5["rank"] = top5.groupby("cluster").cumcount() + 1
view = top5[["cluster", "rank", "sae_feature", "label", "category", "scores", "idf", "score_idf"]]

def _shade(row):
    tint = "#eef3fa" if row.name[0] % 2 == 0 else "#ffffff"
    return [f"background-color: {tint}"] * len(row)

styled = (view.set_index(["cluster", "rank"]).style
              .format({"scores": "{:.1f}", "idf": "{:.2f}", "score_idf": "{:.1f}"})
              .apply(_shade, axis=1)
              .set_properties(**{"text-align": "left"})
              .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))
with pd.option_context("display.max_rows", None, "display.max_colwidth", 60):
    display(styled)

In [ ]:
# Rich context for each cluster's lead (IDF-weighted top) feature
lead = top5[top5["rank"] == 1].sort_values("cluster")
for r in lead.itertuples():
    ap = (str(r.activation_pattern) or "").strip().replace("\n", " ")
    ex = (str(r.exemplar_protein_families) or "").strip().splitlines()
    print(f"\u2501\u2501 cluster {r.cluster}  [{r.sae_feature}] {r.label}  ({r.category})")
    print(f"     activation : {ap[:220]}")
    print(f"     exemplars  : {(ex[0][:200] if ex else '')}")
    print(f"     SwissProt  : {r.swissprot_top}")

In [ ]:
# Dotplot of marker SAE features across clusters (Wilcoxon ranking)
sc.tl.dendrogram(adata, groupby="leiden")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=5)

## Per-cluster feature profile — salience × ubiquity

A second lens on cluster identity, ranked by **mean normalized activation** (what ESMC finds most *salient* across the cluster) with **occurrence** = the fraction of members in which the feature is active. Universal features (occurrence ≈ 100%) define the family; partial ones (≈ 30–60%) flag subgroups or domain variants. This complements the differential Wilcoxon markers above. Reporting the **top 10** per cluster — ranks 6–15 are where subfamily discrimination lives.

> **Residue regions** (start/end/peak per feature) would be the next upgrade, but our cache max-pools SAE activations to one value per protein — positions were discarded. Recovering them needs a per-residue SAE re-run (Biohub credits).

In [ ]:
from och_annotate.analysis import cluster_feature_profile

# Top-10 features per cluster by mean normalized activation (+ occurrence rate)
profile = cluster_feature_profile(adata, groupby="leiden", n=10)

# Atlas labels/category for the profile features (cached; no Biohub credits)
pf_ids = sorted(profile["sae_feature"].unique())
pmeta = fetch_feature_descriptions(pf_ids, cache_path="../data/sae_feature_metadata.parquet")
plabel = dict(zip(pmeta["feature"].astype(int), pmeta["label"]))
pcat = dict(zip(pmeta["feature"].astype(int), pmeta["category"]))
profile["label"] = profile["sae_feature"].map(plabel)
profile["category"] = profile["sae_feature"].map(pcat)
profile.to_csv("../data/cluster_feature_profile_fullsae.csv", index=False)
print(f"Profiled {profile['cluster'].nunique()} clusters x top-10 features "
      f"({len(pf_ids)} unique features)")

In [ ]:
# Grouped top-10 profile per cluster: salience (mean_activation) + ubiquity (occurrence)
pv = (profile.assign(cluster=lambda d: d["cluster"].astype(int))
             .sort_values(["cluster", "rank"])
             [["cluster", "rank", "sae_feature", "label", "category",
               "mean_activation", "occurrence"]])

def _shade2(row):
    tint = "#eef3fa" if row.name[0] % 2 == 0 else "#ffffff"
    return [f"background-color: {tint}"] * len(row)

styled = (pv.set_index(["cluster", "rank"]).style
            .format({"mean_activation": "{:.3f}", "occurrence": "{:.0%}"})
            .apply(_shade2, axis=1)
            .set_properties(**{"text-align": "left"})
            .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))
with pd.option_context("display.max_rows", None, "display.max_colwidth", 60):
    display(styled)

## Per-candidate feature report

The per-protein workflow: **top-10 normalized features** with Atlas labels, plus **residue regions** (start–end, peak) for the architecture-bearing top few. The `residues` column is wired but blank until a per-residue SAE run populates it (`sae.residue_regions: true` — same Biohub call, no extra cost; needs a re-run).

In [ ]:
from och_annotate.analysis import candidate_feature_report
from och_annotate.atlas import fetch_all_features

fd = fetch_all_features(cache_path="../data/sae_feature_dictionary.parquet")
full_labels = dict(zip(fd["feature"].astype(int), fd["label"]))

cand = df.iloc[0]   # example candidate; swap in any row / transcript_id
rep = candidate_feature_report(cand["sae_top_features"], labels=full_labels, n=10)
print(f"Candidate {cand.get('transcript_id','?')}  ({cand.get('Ochierchiae_name','')})")
display(rep)
print("residue regions:", "present" if rep["residues"].notna().any()
      else "pending a per-residue SAE run (set sae.residue_regions=true)")

### Notes on the Atlas annotations

All metadata comes from the public **ESM Atlas** feature API
(`biohub.ai/esm/protein/api/v1alpha1/features/{idx}`) via
`och_annotate.atlas.fetch_feature_descriptions` — cached under `data/`, **not** charged
against Biohub embedding credits. The grouped table is ranked by
`score_idf = wilcoxon_score × uniref90_idf` so cluster-specific (rare) features rise above
ubiquitous ones; the lead-feature block adds activation pattern, exemplar families and
reviewed-UniProt examples. Full table: `data/cluster_sae_enrichment_saebasis.csv`.

### Other next steps
- Write `adata.obs["leiden"]` back to Baserow as a `leiden_cluster` column.
- GO-enrich each cluster from the SwissProt example proteins.
- Tune `LEIDEN_RES`, `N_NEIGHBORS`, `MIN_DIST`.